# Bank Marketing Decision Lab — Research Exploration

**Researcher:** Jinlei Cao  
**Data:** UCI Bank Marketing, `bank-additional-full.csv` (Moro et al., 2014)  
**Academic integrity:** The default mode is synthetic; use genuine results only after running the real dataset.  

The dataset is ordered approximately chronologically. We split chronologically, exclude post-call `duration`, and compare simple predictive models. **Predicted subscription is not causal uplift from a campaign call.**

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from banklab.data import load_data, download_uci, make_synthetic_demo
from banklab.features import prepare_features, chronological_indices

USE_SYNTHETIC = True  # Set to False only when ready to download and study real UCI data
if USE_SYNTHETIC:
    df = make_synthetic_demo(n=1_600, seed=42)
    print('SYNTHETIC DEMO: not empirical research findings')
else:
    path = download_uci('data/raw/bank-additional-full.csv')
    df = load_data(path)
    print('Loaded real UCI historical campaign data')
print('Records:', len(df), '| Observed subscription rate:', round(df.y.eq('yes').mean(), 4))
df.head()

## Exploratory analysis

Inspect target prevalence and descriptive associations. The raw dataset may contain multiple contacts for the same customer and cannot identify untreated counterfactual outcomes.

In [ ]:
print(df[['age', 'campaign', 'previous']].describe())
print('\nSubscription by contact channel:')
print(df.groupby('contact')['y'].apply(lambda s: s.eq('yes').mean()).sort_values(ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['age'].hist(bins=25, ax=axes[0])
axes[0].set(title='Age distribution', xlabel='Age')
df['y'].value_counts().plot.bar(ax=axes[1])
axes[1].set(title='Subscription outcome', ylabel='Contact count')
plt.tight_layout()
plt.show()

## Validate feature eligibility and chronological boundaries

`duration` and `y` must be excluded from predictors. UCI's `pdays == 999` means no prior contact.

In [ ]:
X, y = prepare_features(df)
assert 'duration' not in X and 'y' not in X
train_idx, valid_idx, test_idx = chronological_indices(len(df))
print('Predictor count:', X.shape[1])
print('Train / validation / test:', len(train_idx), len(valid_idx), len(test_idx))
print('Positive rate by period:', [round(y.iloc[i].mean(), 4) for i in (train_idx, valid_idx, test_idx)])

## Run reproducible experiment

The next cell trains two models, selects the champion using *validation average precision*, and writes final-holdout plots and a report. Results in synthetic mode are only engineering smoke tests.

In [ ]:
from banklab.experiment import run_experiment
output = 'reports/notebook_demo' if USE_SYNTHETIC else 'reports/notebook_real'
source = 'SYNTHETIC DEMO — NOT EMPIRICAL EVIDENCE' if USE_SYNTHETIC else 'UCI bank-additional-full.csv (DOI:10.24432/C5K306)'
manifest = run_experiment(df, output, source=source, fast=USE_SYNTHETIC)
print('Champion:', manifest['champion'])
print(Path(output, 'RESULTS.md').read_text())

## Interpretation questions

1. How does class prevalence change across chronological partitions?
2. Does the random forest improve *validation AP*, and does the advantage persist on the test set?
3. What proportion of observed subscriptions fall within the top 20% ranked contacts?
4. Which limitations prevent interpreting the results as incremental marketing effects?
5. How could a randomized no-call control group support uplift estimation?

**References:** Moro, S., Rita, P., & Cortez, P. (2014). *Bank Marketing* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306